# Lensless Imaging — Bonus Demo (FISTA + fine-tuned Real-ESRGAN)

This notebook runs the two bonus reconstruction methods on the same dataset and compares them:

- FISTA non-ADMM solver 
- Real-ESRGAN (fine-tuned) pretrained GAN, fine-tuned on ADMM inversion

**How to use:**

1. Fill in `DATA_URL` 
2. Run all cells top to bottom (`Runtime -> Run all`)
3. The notebook downloads the checkpoint and the dataset, runs both methods, visualizes them side by side, and prints the metrics

Designed to work in a Google Colab

## 1. Configuration

- `DATA_URL` — Google Drive link to a `.zip` dataset (structure shown in step 5)
- `REALESRGAN_FT_URL` — Google Drive link (or id) of the fine-tuned Real-ESRGAN `model_best.pth`. FISTA needs no checkpoint
- `ADMM_ITERS` — number of ADMM iterations the Real-ESRGAN checkpoint was trained with (must match the checkpoint)

In [ ]:
REPO_URL = "https://github.com/danyatennn/Lensless_Imaging"
DATA_URL = "https://drive.google.com/file/d/1Z2Pos-4maZDeidN3BfhNZhDdaM8rmyyF/view?usp=sharing"
REALESRGAN_FT_URL = "https://drive.google.com/file/d/1d8GskbAwDphGggOdhIuiHuLHGEDZmV1I/view?usp=sharing" 
ADMM_ITERS = 20 

## 2. Clone the repository

In [ ]:
import os, sys, subprocess

if not os.path.isdir("repo"):
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
if os.path.basename(os.getcwd()) != "repo":
    os.chdir("repo")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Download the fine-tuned Real-ESRGAN checkpoint

Saved to `saved/realesrgan_ft/model_best.pth`. FISTA doesnrt need checkpoint

In [ ]:
import gdown

ESRGAN_CKPT = "saved/realesrgan_ft/model_best.pth"
os.makedirs(os.path.dirname(ESRGAN_CKPT), exist_ok=True)
if not os.path.exists(ESRGAN_CKPT):
    gdown.download(REALESRGAN_FT_URL, ESRGAN_CKPT, quiet=False, fuzzy=True)
print("checkpoint:", ESRGAN_CKPT, "->", round(os.path.getsize(ESRGAN_CKPT) / 1e6, 1), "MB")

## 5. Download and unzip the dataset

 a directory of the form:

```
data
├── lensless        # lensless measurements (required)
│   └── ID.png
├── masks           # mask patterns (required)
│   └── ID.npy
└── lensed          # ground-truth images (optional)
    └── ID.png
```

In [ ]:
import zipfile
from pathlib import Path

if not os.path.exists("dataset.zip"):
    gdown.download(DATA_URL, "dataset.zip", quiet=False, fuzzy=True)
with zipfile.ZipFile("dataset.zip") as archive:
    archive.extractall("demo_data")

DATA_DIR = next(p.parent for p in Path("demo_data").rglob("lensless") if p.is_dir())
FISTA_DIR = Path("reconstructions_fista").absolute()
ESRGAN_DIR = Path("reconstructions_realesrgan_ft").absolute()
print("data dir:", DATA_DIR)
print("FISTA ->", FISTA_DIR)
print("Real-ESRGAN ft ->", ESRGAN_DIR)

## 6. Run inference for both bonus methods


In [ ]:
!python inference.py \
    model=fista \
    inferencer.from_pretrained=null \
    datasets=custom_dir \
    datasets.test.data_dir="$DATA_DIR" \
    inferencer.save_path="$FISTA_DIR"

!python inference.py \
    model=admm100_realesrgan_ft \
    model.camera_inversion.n_iters=$ADMM_ITERS \
    inferencer.from_pretrained="$ESRGAN_CKPT" \
    datasets=custom_dir \
    datasets.test.data_dir="$DATA_DIR" \
    inferencer.save_path="$ESRGAN_DIR"

## 7. Visualize and compare

Original (if available) vs. lensless measurement vs. FISTA vs. fine-tuned Real-ESRGAN.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sample_ids = [p.stem for p in sorted(FISTA_DIR.glob("*.png"))[:4]]
lensless_dir = DATA_DIR / "lensless"
lensed_dir = DATA_DIR / "lensed"

titles = ["original (lensed)", "lensless", "FISTA", "Real-ESRGAN (ft)"]
n = len(sample_ids)
fig, axes = plt.subplots(n, 4, figsize=(12, 3 * n))
axes = np.atleast_2d(axes)
for row, sample_id in enumerate(sample_ids):
    lensless = np.asarray(
        Image.open(lensless_dir / f"{sample_id}.png").convert("RGB")
    ).astype(np.float32)
    lensless = lensless / (lensless.max() + 1e-8)

    lensed_path = lensed_dir / f"{sample_id}.png"
    if lensed_path.exists():
        axes[row, 0].imshow(Image.open(lensed_path).convert("RGB"))
    axes[row, 1].imshow(lensless)
    axes[row, 2].imshow(Image.open(FISTA_DIR / f"{sample_id}.png").convert("RGB"))
    axes[row, 3].imshow(Image.open(ESRGAN_DIR / f"{sample_id}.png").convert("RGB"))
    for col, title in enumerate(titles):
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## 8. Compute metrics

If the dataset has ground-truth `lensed` images, print PSNR / SSIM / LPIPS / MSE for each method.

In [ ]:
print("=== FISTA ===")
!python calculate_metrics.py --ground_truth "$DATA_DIR" --reconstructions "$FISTA_DIR"
print("\n=== Real-ESRGAN (fine-tuned) ===")
!python calculate_metrics.py --ground_truth "$DATA_DIR" --reconstructions "$ESRGAN_DIR"